# 01 — Load the model-training dataset

This notebook loads and validates the frozen files used for model training. It does not call live APIs, train a model, or use deferred source-selection data.

The prepared target is Belgium's next-hour average photovoltaic generation estimate in **MW**. Data is split chronologically into training, validation, and an untouched 2025 test set.

Regenerate the files only from the frozen raw inputs:

```bash
uv run python -m scripts.prepare_training_data --force
```

## 1. Setup and required files

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter inside the Solar_forecast repository."
    )


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


ROOT = find_repository_root()
TRAINING_ROOT = ROOT / "data" / "processed" / "model_training"
DATASET_MANIFEST_PATH = TRAINING_ROOT / "dataset_manifest.json"
ELIA_ROOT = ROOT / "data" / "raw" / "elia_generation"
ELIA_DATA_PATH = ELIA_ROOT / "belgium_2024_2025.csv"
ELIA_MANIFEST_PATH = ELIA_ROOT / "source_manifest.json"
WEATHER_ROOT = ROOT / "data" / "raw" / "belgium_weather_archive"
WEATHER_MANIFEST_PATH = WEATHER_ROOT / "download_manifest.json"

required_paths = [
    DATASET_MANIFEST_PATH,
    ELIA_DATA_PATH,
    ELIA_MANIFEST_PATH,
    WEATHER_MANIFEST_PATH,
]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    missing_text = "\n".join(f"- {path.relative_to(ROOT)}" for path in missing_paths)
    raise FileNotFoundError(f"Required training data is missing:\n{missing_text}")

print(f"Repository root: {ROOT}")
print(f"pandas version: {pd.__version__}")

## 2. Verify raw-source provenance

The processed dataset records source hashes. These checks ensure the retained Elia target and Belgian weather manifest are the exact inputs used during preparation.

In [ ]:
with DATASET_MANIFEST_PATH.open(encoding="utf-8") as source:
    dataset_manifest = json.load(source)
with ELIA_MANIFEST_PATH.open(encoding="utf-8") as source:
    elia_manifest = json.load(source)

source_checks = pd.DataFrame(
    [
        {
            "source": "Elia generation CSV",
            "path": str(ELIA_DATA_PATH.relative_to(ROOT)),
            "bytes_match": ELIA_DATA_PATH.stat().st_size
            == elia_manifest["file"]["bytes"],
            "hash_matches_source_manifest": sha256_file(ELIA_DATA_PATH)
            == elia_manifest["file"]["sha256"],
            "hash_matches_dataset_manifest": sha256_file(ELIA_DATA_PATH)
            == dataset_manifest["audit"]["elia"]["sha256"],
        },
        {
            "source": "Belgian weather manifest",
            "path": str(WEATHER_MANIFEST_PATH.relative_to(ROOT)),
            "bytes_match": True,
            "hash_matches_source_manifest": True,
            "hash_matches_dataset_manifest": sha256_file(WEATHER_MANIFEST_PATH)
            == dataset_manifest["audit"]["weather"]["weather_manifest_sha256"],
        },
    ]
)
if not source_checks[
    ["bytes_match", "hash_matches_source_manifest", "hash_matches_dataset_manifest"]
].all().all():
    raise ValueError("A retained raw source differs from the prepared dataset manifest.")
source_checks

## 3. Load and verify train, validation, and test files

Every file is verified by byte size and SHA-256 before loading. The target, timestamp, and split columns are not model features.

In [ ]:
target_column = dataset_manifest["target"]["column"]
feature_columns = dataset_manifest["feature_columns"]
expected_columns = ["timestamp", "split", target_column, *feature_columns]
splits: dict[str, pd.DataFrame] = {}

for entry in dataset_manifest["files"]:
    path = TRAINING_ROOT / entry["file"]
    if not path.is_file():
        raise FileNotFoundError(f"Prepared split is missing: {path}")
    if path.stat().st_size != entry["bytes"] or sha256_file(path) != entry["sha256"]:
        raise ValueError(f"Prepared split differs from its manifest: {path.name}")

    frame = pd.read_csv(path)
    frame["timestamp"] = pd.to_datetime(
        frame["timestamp"], format="%Y-%m-%dT%H:%M:%SZ", errors="raise", utc=True
    )
    if list(frame.columns) != expected_columns:
        raise ValueError(f"Unexpected schema in {path.name}.")
    if len(frame) != entry["rows"]:
        raise ValueError(f"Unexpected row count in {path.name}.")
    if set(frame["split"].unique()) != {entry["split"]}:
        raise ValueError(f"Split label mismatch in {path.name}.")
    if frame.isna().any().any() or frame["timestamp"].duplicated().any():
        raise ValueError(f"Missing values or duplicate timestamps in {path.name}.")
    numeric = frame[[target_column, *feature_columns]].to_numpy(dtype="float64")
    if not np.isfinite(numeric).all():
        raise ValueError(f"Non-finite numeric value in {path.name}.")
    if not frame["timestamp"].is_monotonic_increasing:
        raise ValueError(f"Timestamps are not chronological in {path.name}.")
    splits[entry["split"]] = frame

train_data = splits["train"]
validation_data = splits["validation"]
test_data = splits["test"]

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(frame),
            "first_timestamp_utc": frame["timestamp"].min(),
            "last_timestamp_utc": frame["timestamp"].max(),
            "columns": len(frame.columns),
            "missing_cells": int(frame.isna().sum().sum()),
        }
        for name, frame in splits.items()
    ]
)
split_summary

In [ ]:
if not (
    train_data["timestamp"].max() < validation_data["timestamp"].min()
    < test_data["timestamp"].min()
):
    raise ValueError("Chronological split boundaries overlap or are out of order.")
if target_column in feature_columns:
    raise ValueError("Target leakage: the target appears in feature_columns.")
if {"timestamp", "split"} & set(feature_columns):
    raise ValueError("Metadata columns appear in feature_columns.")

pd.Series(
    {
        "schema_version": dataset_manifest["schema_version"],
        "target_column": target_column,
        "target_unit": dataset_manifest["target"]["unit"],
        "features": len(feature_columns),
        "total_rows": sum(len(frame) for frame in splits.values()),
        "chronological_boundaries_valid": True,
        "target_excluded_from_features": True,
    },
    name="value",
).to_frame()

## 4. Feature groups

In [ ]:
feature_groups = {
    "capacity": [column for column in feature_columns if column == "monitored_capacity_mw"],
    "temperature": [column for column in feature_columns if column.startswith("temperature_")],
    "cloud_cover": [column for column in feature_columns if column.startswith("cloud_cover_")],
    "radiation": [
        column for column in feature_columns if column.startswith("shortwave_radiation_")
    ],
    "temporal": [column for column in feature_columns if column.startswith("target_")],
    "generation_lags": [
        column for column in feature_columns if column.startswith("power_generation_lag_")
    ],
}
grouped_features = {column for columns in feature_groups.values() for column in columns}
if grouped_features != set(feature_columns):
    raise ValueError("At least one model feature is unclassified or classified twice.")

pd.DataFrame(
    {
        "feature_group": feature_groups.keys(),
        "feature_count": [len(columns) for columns in feature_groups.values()],
    }
)

## 5. Training and validation target inspection

The final 2025 test target is intentionally not summarized here. Do not use the test set for feature decisions, model selection, thresholds, or hyperparameter tuning.

In [ ]:
pd.DataFrame(
    {
        "train": train_data[target_column].describe(),
        "validation": validation_data[target_column].describe(),
    }
)

In [ ]:
train_data.head()

## 6. Objects ready for model training

The cells below create feature matrices and targets without fitting anything. Keep `X_test` and `y_test` untouched until the model family and hyperparameters have been finalized using only training and validation data.

In [ ]:
X_train = train_data[feature_columns].copy()
y_train = train_data[target_column].copy()
X_validation = validation_data[feature_columns].copy()
y_validation = validation_data[target_column].copy()
X_test = test_data[feature_columns].copy()
y_test = test_data[target_column].copy()

pd.DataFrame(
    {
        "object": ["X_train", "y_train", "X_validation", "y_validation", "X_test", "y_test"],
        "shape": [
            X_train.shape,
            y_train.shape,
            X_validation.shape,
            y_validation.shape,
            X_test.shape,
            y_test.shape,
        ],
    }
)

## Result

The model-training data is loaded, checksummed, finite, missing-value free, chronologically split, and separated into `X`/`y` objects. Deferred live/API and alternative-source data is documented in `docs/DEFERRED_DATA_SOURCES.md` and is not required for this training phase.